# LeIsaac VLA Play

来源：<https://lightwheelai.github.io/leisaac/resources/available_policy/>

**前提条件**（运行任何 cell 之前）：
1. 在 `isaaclab-experience/` 目录下启动 jupyter
2. 启动前已激活 conda 环境：`conda activate isaaclab`

注意：先启动 policy server，再运行推理。一次只跑一个推理单元。
如需可视化窗口，请不要加 `--headless`（本 notebook 默认已移除）。无显示环境再手动添加。

## 0) 预检查

In [ ]:
!cd LeIsaac && test -f scripts/evaluation/policy_inference.py && test -f assets/robots/so101_follower.usd && test -f assets/scenes/kitchen_with_orange/scene.usd && python scripts/evaluation/policy_inference.py --help | head -n 20

## 1) 服务端一键后台启动（推荐）

直接调用：`./server/start_server.sh [--gr00t-only|--lerobot-only]` / `./server/status_server.sh` / `./server/stop_server.sh`

In [ ]:
!GR00T_SIM_WRAPPER=1 bash server/start_server.sh


In [ ]:
!bash server/status_server.sh

In [ ]:
!bash server/stop_server.sh

## 2) GR00T N1.6 推理（GR1 双臂人形）

**模型适配场景**：`robocasa-gr1-tabletop`（24 个 PnP 任务，base 模型自带 GR1 head，可直接 zero-shot）

**不适配场景**：LeIsaac SO-101 系列 —— base 模型没有 SO-101 head，必须先 finetune。

## 2.1) 检查并启动 GR00T 推理服务


In [ ]:
!bash scripts/check_start_gr00t.sh


## 2.2) 实时预览推理（GR1 双臂 tabletop）

跑通后 `totem` 自动全屏播放生成的 mp4。任务可用 `ENV_NAME=...` 覆盖。


In [ ]:
!bash scripts/preview_gr00t_inference.sh


### ⚠ 注意：GR00T base 不能直接推理 LeIsaac SO-101

`GR00T-N1.6-3B` 内置的 EmbodimentTag 是 `GR1 / UNITREE_G1 / ROBOCASA_PANDA_OMRON / LIBERO_PANDA / OXE_GOOGLE / OXE_WIDOWX / OXE_DROID / BEHAVIOR_R1_PRO`，**没有 SO-101**。

要在 LeIsaac SO-101 任务上用 GR00T 推理，必须：
1. 在 SO-101 上采集（或获取）demonstration 数据，转成 LeRobot V2 schema；
2. 按 [`getting_started/finetune_new_embodiment.md`](../Isaac-GR00T/getting_started/finetune_new_embodiment.md) 用 `NEW_EMBODIMENT` tag finetune；
3. 用 finetuned checkpoint 重启 server（**去掉** `--use-sim-policy-wrapper`，那是给 robocasa 框架用的）；
4. 再回来跑 LeIsaac SO-101 推理。

本 cell **不可执行**。要看 GR00T 真跑机器人的效果，请回到 §2.2。

## 3) LeRobot SmolVLA 推理（LeIsaac SO-101 单臂）

**模型适配场景**：LeIsaac-SO101-* 系列任务（如 PickOrange、LiftCube、CleanToyTable…）

SmolVLA `lerobot/smolvla_base` 是 HuggingFace LeRobot 团队发布的通用单臂 manipulation VLA，训练分布中包含 SO-100/SO-101 真机数据，**可直接 zero-shot 推理 LeIsaac 仿真中的 SO-101**。

### 3.1) 启动 LeRobot 推理服务

端口 `:8080`。已起则 idempotent skip。


In [ ]:
!bash server/start_server.sh --lerobot-only


### 3.2) 安装 LeIsaac LeRobot client 依赖（首次执行后可跳过）


In [ ]:
!cd LeIsaac && pip install -e "source/leisaac[lerobot-async]"


### 3.3) 运行 SO-101 PickOrange 仿真推理

Isaac Sim 会弹窗显示 SO-101 单臂；客户端连 `:8080` 调 SmolVLA 推理。


In [ ]:
!cd LeIsaac && python scripts/evaluation/policy_inference.py --task=LeIsaac-SO101-PickOrange-v0 --eval_rounds=1 --policy_type=lerobot-smolvla --policy_host=127.0.0.1 --policy_port=8080 --policy_timeout_ms=15000 --policy_language_instruction='Pick the orange to the plate' --policy_checkpoint_path=lerobot/smolvla_base --policy_action_horizon=16 --device=cuda --enable_cameras


### 3.4) 停止 LeRobot 推理服务（释放显存）

不影响 GR00T `:5555`。


In [ ]:
!bash server/stop_server.sh --lerobot-only
